# NHESS Paper Abstract Scraper

This notebook scrapes the latest 1-2 final revised papers from NHESS and saves cleaned metadata to `arxiv_clean.json`.

Note: the assignment wording mentions `cs.CL` and `/abs/`, which are arXiv terms. This version follows the requested NHESS journal target while keeping the required output filename.

## 1. Imports and settings

In [3]:
import json
import re
from pathlib import Path
from typing import Any
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup

RECENT_URL = "https://nhess.copernicus.org/recent_papers.html"
OUTPUT_DIR = Path.cwd() / "hw_output" / "Paper_Abstract"
OUTPUT_PATH = OUTPUT_DIR / "arxiv_clean.json"
MAX_PAPERS = 2
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; course-abstract-scraper/1.0)"}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 2. Scraper helpers

In [4]:
def clean_text(text: str) -> str:
    text = re.sub(r"\s+", " ", text).strip()
    return text.removeprefix("Abstract ").strip()


def fetch_soup(url: str) -> BeautifulSoup:
    response = requests.get(url, headers=HEADERS, timeout=30)
    response.raise_for_status()
    response.encoding = "utf-8"
    return BeautifulSoup(response.text, "html.parser")


def first_meta(soup: BeautifulSoup, name: str) -> str:
    tag = soup.find("meta", attrs={"name": name})
    return clean_text(tag["content"]) if tag and tag.get("content") else ""


def all_meta(soup: BeautifulSoup, name: str) -> list[str]:
    return [clean_text(tag["content"]) for tag in soup.find_all("meta", attrs={"name": name}) if tag.get("content")]

## 3. Find recent NHESS final papers

In [5]:
def recent_final_article_urls(limit: int = MAX_PAPERS) -> list[str]:
    soup = fetch_soup(RECENT_URL)
    urls: list[str] = []
    for item in soup.select(".paperList-final"):
        title_link = item.find("a", href=re.compile(r"/articles/\d+/\d+/\d+/$"))
        if not title_link:
            continue
        url = urljoin(RECENT_URL, title_link["href"])
        if url not in urls:
            urls.append(url)
        if len(urls) >= limit:
            break
    return urls


article_urls = recent_final_article_urls(MAX_PAPERS)
article_urls

['https://nhess.copernicus.org/articles/26/2525/2026/',
 'https://nhess.copernicus.org/articles/26/2505/2026/']

## 4. Parse article pages and save JSON

In [6]:
def parse_article(url: str) -> dict[str, Any]:
    soup = fetch_soup(url)
    abstract_node = soup.select_one("#abstract, div.abstract, .abstract")
    abstract = clean_text(abstract_node.get_text(" ", strip=True)) if abstract_node else ""

    date = first_meta(soup, "citation_publication_date").replace("/", "-")
    return {
        "url": url,
        "title": first_meta(soup, "citation_title"),
        "abstract": abstract,
        "authors": all_meta(soup, "citation_author"),
        "date": date,
    }


records = [parse_article(url) for url in article_urls]
OUTPUT_PATH.write_text(json.dumps(records, ensure_ascii=False, indent=2), encoding="utf-8")

size = OUTPUT_PATH.stat().st_size
if size > 1_000_000:
    raise RuntimeError(f"{OUTPUT_PATH} is larger than 1 MB: {size} bytes")

print(f"Wrote {len(records)} records to {OUTPUT_PATH} ({size} bytes)")
records

Wrote 2 records to /Users/ming/Desktop/Learning/MLE_in_Gen_AI-Course/class2/hw_output/Paper_Abstract/hw_output/Paper_Abstract/arxiv_clean.json (3818 bytes)


[{'url': 'https://nhess.copernicus.org/articles/26/2525/2026/',
  'title': 'Multi-level assessment of flood risk perception and flood behaviour',
  'abstract': 'Understanding the relationships between flood risk perception and flood behaviour is crucial for effective risk management and risk communication strategies, but quantitative research in this area remains challenging. Based on a survey of 1007 residents in four different localities of Chile exposed to river floods, this study builds and applies a framework for assessing flood risk perception and flood behaviour at the individual, household, neighbourhood, and municipality levels. Results show that almost all respondents were aware of flood risk. Economic and personal resources strongly influence worry and preparedness: households in better economic situations were less worried about floods, lower economic resources at the municipal and neighbourhood levels prompted households to adopt cautionary measures. Experiences where the 

## 5. Standalone script

The same scraper is also saved as `scrape_nhess_abstracts.py` for submission.